<a href="https://colab.research.google.com/github/Bvsrkrishna/Data-Analyst-and-Engineering/blob/main/Analysis%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PipelinAnalytics

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import sys, os
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False  #@param {type:"boolean"}
if IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/clv_project")   # files survive between sessions
elif IN_COLAB:
    BASE = Path("/content/clv_project")                                                                              # lost when runtime resets
else:
    BASE = Path.cwd() / "clv_project"
BASE.mkdir(parents=True, exist_ok=True); os.chdir(BASE)
print("Working folder:", BASE)
REQUIRED = ['data/processed/transactions_clean.csv', 'data/processed/customers_clean.csv', 'data/processed/products_clean.csv', 'data/processed/rfm.csv']
missing = [p for p in REQUIRED if not (BASE / p).exists()]

Working folder: /content/clv_project


In [ ]:
import json, warnings
import numpy as np, pandas as pd, matplotlib

import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")


PROC, FIG = BASE / "data" / "processed", BASE / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
CUTOFF, HORIZON_DAYS, SEED = pd.Timestamp("2025-06-30"), 183, 42

tx = pd.read_csv(PROC / "transactions_clean.csv", parse_dates=["order_date"])
cu = pd.read_csv(PROC / "customers_clean.csv", parse_dates=["signup_date"])
pr = pd.read_csv(PROC / "products_clean.csv")
tx = tx.merge(pr[["product_id", "category"]], on="product_id")
CAT = ["country", "acquisition_channel"]

### feature builder

In [ ]:
# ------------------------------------------------------------ feature builder
def build_features(tx, asof):
    h = tx[tx.order_date <= asof]
    o = h.groupby(["order_id", "customer_id", "order_date"], as_index=False).line_revenue.sum()
    g = o.groupby("customer_id")
    f = pd.DataFrame({
        "frequency": g.order_id.nunique(), "monetary": g.line_revenue.sum(), "aov": g.line_revenue.mean(),
        "recency_days": (asof - g.order_date.max()).dt.days,
        "tenure_days": (asof - g.order_date.min()).dt.days})
    o = o.sort_values(["customer_id", "order_date"])
    gap = o.groupby("customer_id").order_date.diff().dt.days
    f["avg_gap_days"] = gap.groupby(o.customer_id).mean().reindex(f.index).fillna(f.tenure_days.clip(lower=1))
    f["recency_ratio"] = f.recency_days / f.avg_gap_days.clip(lower=1)     # "overdue-ness"
    for d in (90, 180):
        r = o[o.order_date > asof - pd.Timedelta(days=d)].groupby("customer_id")
        f[f"orders_last_{d}d"] = r.order_id.nunique().reindex(f.index).fillna(0)
        f[f"rev_last_{d}d"] = r.line_revenue.sum().reindex(f.index).fillna(0)
    f["n_categories"] = h.groupby("customer_id").category.nunique()
    f["avg_discount"] = h.groupby("customer_id").discount.mean()
    f["app_share"] = h.assign(a=h.order_channel.eq("App")).groupby("customer_id").a.mean()
    f = f.reset_index().merge(cu[["customer_id", "age", "country", "acquisition_channel"]], on="customer_id")
    return pd.get_dummies(f, columns=CAT, drop_first=True, dtype=float)

def target(tx, start, days):
    t = tx[(tx.order_date > start) & (tx.order_date <= start + pd.Timedelta(days=days))]
    return t.groupby("customer_id").line_revenue.sum()

### train / evaluate

In [ ]:
# ------------------------------------------------------------ train / evaluate
X = build_features(tx, CUTOFF)
X["y"] = X.customer_id.map(target(tx, CUTOFF, HORIZON_DAYS)).fillna(0)
feats = [c for c in X.columns if c not in ("customer_id", "y")]
tr, te = train_test_split(X, test_size=0.25, random_state=SEED)
print(f"customers={len(X):,} | train={len(tr):,} test={len(te):,} | avg 6m revenue=${X.y.mean():,.0f} "
      f"| active in holdout={(X.y > 0).mean():.1%}")

def metrics(name, y, p):
    m = {"model": name, "MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** .5,
         "R2": r2_score(y, p), "Spearman": spearmanr(y, p)[0]}
    top = np.argsort(-np.asarray(p))[:max(1, len(y) // 10)]
    m["Top10%_capture"] = float(np.asarray(y)[top].sum() / np.sum(y))      # share of true revenue found in top decile
    return m

res = []
base_p = te.monetary / (te.tenure_days.clip(lower=30) / 30) * 6          # naive: historical monthly spend x 6
res.append(metrics("Baseline (hist. run-rate)", te.y, base_p))

try:                                                                     # statsmodels OLS
    import statsmodels.api as sm
    ols = sm.OLS(tr.y, sm.add_constant(tr[feats])).fit()
    p = ols.predict(sm.add_constant(te[feats], has_constant="add")).clip(lower=0)
    res.append(metrics("OLS (statsmodels)", te.y, p))
    pd.DataFrame({"coef": ols.params, "p_value": ols.pvalues}).round(4).to_csv(PROC / "ols_summary.csv")
except ImportError:
    print("statsmodels not installed -> skipping OLS (pip install statsmodels)")

gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8,
                                loss="huber", random_state=SEED).fit(tr[feats], tr.y)
pg = np.clip(gbr.predict(te[feats]), 0, None)
res.append(metrics("GradientBoosting (sklearn)", te.y, pg))
rep = pd.DataFrame(res).round(3); print("\n", rep.to_string(index=False))
rep.to_csv(PROC / "model_metrics.csv", index=False)

### charts

In [ ]:
# ------------------------------------------------------------ charts
imp = pd.Series(gbr.feature_importances_, feats).sort_values().tail(12)
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
imp.plot.barh(ax=axs[0], title="Top feature importances")
axs[1].scatter(te.y, pg, s=6, alpha=.4); lim = max(te.y.max(), pg.max())
axs[1].plot([0, lim], [0, lim], "r--"); axs[1].set(title="Actual vs predicted 6m revenue", xlabel="actual", ylabel="predicted")
fig.tight_layout(); fig.savefig(FIG / "07_model_diagnostics.png", dpi=130); plt.show()

### final scoring (as of last data date)

In [ ]:
# ------------------------------------------------------------ final scoring (as of last data date)
ASOF = tx.order_date.max()
final = build_features(tx, ASOF)
for c in feats:                                   # align dummy columns
    if c not in final: final[c] = 0.0
final["predicted_clv_6m"] = np.clip(gbr.predict(final[feats]), 0, None).round(2)
final["total_clv"] = (final.monetary + final.predicted_clv_6m).round(2)
q = final.predicted_clv_6m.rank(pct=True)
final["clv_segment"] = np.select([q > .9, q > .7, q > .4], ["Platinum", "Gold", "Silver"], "Bronze")

keep = ["customer_id", "recency_days", "frequency", "monetary", "aov", "tenure_days", "predicted_clv_6m", "total_clv", "clv_segment"]
out = final[keep].merge(cu[["customer_id", "country", "acquisition_channel", "age", "gender", "signup_date"]], on="customer_id")
rfm = pd.read_csv(PROC / "rfm.csv")[["customer_id", "rfm_segment"]]
out = out.merge(rfm, on="customer_id", how="left")
out.to_csv(PROC / "clv_predictions.csv", index=False)
seg = out.groupby("clv_segment").agg(customers=("customer_id", "count"), pred_clv=("predicted_clv_6m", "sum"),
                                     avg_clv=("predicted_clv_6m", "mean"), hist_rev=("monetary", "sum")).round(0)
print("\nSegment summary\n", seg.sort_values("avg_clv", ascending=False))
json.dump({"cutoff": str(CUTOFF.date()), "scored_asof": str(ASOF.date()), "n_customers": len(out)},
          open(PROC / "model_run.json", "w"))

### Download results (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import files
    for f in ['data/processed/clv_predictions.csv', 'data/processed/model_metrics.csv']:
        files.download(str(BASE / f))